# Verification code for *Affine Higher Bruhat Orders* by Sara Billey, Herman Chau, and Kevin Liu

The code uses the parameter `K` for the size of the consistent set: `G(w,K,R)` has vertex set `Inv_{K-1}(w)` and assumes `R` is a subset of `Inv_K(w)`.  The finite reduction checks use `affine=False`; the affine conjecture check uses `affine=True`, which includes congruence edges and affine quasi-inversion edges.


In [ ]:
from __future__ import annotations

from typing import TypeVar, TypedDict, Optional
from functools import lru_cache
from collections import deque
import itertools
import math
import time

T = TypeVar('T')
Perm = tuple[int, ...]
Inv = tuple[int, ...]

class PosetEntry(TypedDict):
    ins: set
    outs: set

Poset = dict[T, PosetEntry]
Graph = Poset[Inv]


def residue_repr(x: int, n: int) -> int:
    """Return the congruence representative of x in the range [1,n]."""
    return n if x % n == 0 else x % n

# Backward-compatible alias for the original notebook's congruence representative.
repr_mod = residue_repr


def inverse(w: Perm) -> Perm:
    """Return the window of the inverse affine permutation w^{-1}."""
    n = len(w)
    w_inv = [0 for _ in range(n)]
    for i in range(n):
        j = residue_repr(w[i], n)
        w_inv[j - 1] = i + 1 + (j - w[i])
    return tuple(w_inv)


def normalize(x: Inv, n: int) -> Inv:
    """Normalize a tuple modulo the diagonal n-shift by putting the first entry in [1,n]."""
    if not x:
        return tuple()
    shift = residue_repr(x[0], n) - x[0]
    return tuple(i + shift for i in x)


@lru_cache(maxsize=None)
def inv(w: Perm, k: int) -> frozenset[Inv]:
    """Return all k-inversions of w, using normalized representatives."""
    if k < 0:
        return frozenset()
    if k == 0:
        return frozenset([tuple()])

    n = len(w)
    if k > n:
        return frozenset()

    w_inv = inverse(w)
    ret: set[Inv] = set()

    def inv_helper(idxs: list[int], vals: list[int]) -> list[list[int]]:
        if len(idxs) == 1:
            return [[idxs[0]]]

        out: list[list[int]] = []
        i = 0
        while vals[0] - vals[1] > i * n:
            new_idxs = [x + i * n for x in idxs[1:]]
            new_vals = [x + i * n for x in vals[1:]]
            for r in inv_helper(new_idxs, new_vals):
                out.append([idxs[0]] + r)
            i += 1
        return out

    for xs in itertools.permutations(range(1, n + 1), k):
        idxs = list(xs)
        vals = [w_inv[i - 1] for i in idxs]

        for i in range(1, k):
            while idxs[i] < idxs[i - 1]:
                idxs[i] += n
                vals[i] += n
        for r in inv_helper(idxs, vals):
            ret.add(tuple(r))
    return frozenset(ret)


def packet(x: Inv, n: int) -> list[Inv]:
    """Return the packet P(x) in antilex order."""
    return [normalize(tuple(x[:i] + x[i + 1:]), n) for i in range(len(x))]


def finite_subsets(n: int, k: int) -> list[Inv]:
    """Classical k-subsets of [n], represented as increasing tuples."""
    if k < 0 or k > n:
        return []
    return list(itertools.combinations(range(1, n + 1), k))


def is_congruent(i: Inv, j: Inv, n: int) -> bool:
    """Return whether i and j represent the same residue set in the same order."""
    if len(i) != len(j):
        return False
    return all(i[t] % n == j[t] % n for t in range(len(i)))


def common_packet_affine(i: Inv, j: Inv, n: int) -> Optional[Inv]:
    """Return X such that i and j both lie in P(X), if such an affine X is found.

    The inputs are normalized representatives of elements of binom(Z,k)_n.
    This is used only for the affine permanent poset with k >= 2 in the
    verification ranges below.  The result is verified by packet membership.
    """
    if len(i) != len(j):
        return None
    k = len(i)
    if k <= 0:
        return None

    candidates: set[int] = set()
    for a in i:
        for b in j:
            if residue_repr(a, n) == residue_repr(b, n):
                candidates.add(a - b)

    # For k=1 the affine permanent poset is infinite; the verification below
    # never uses the affine k=1 model.  Keep a finite fallback only for local
    # packet detection, not for paper-level affine k=1 verification.
    if k == 1 and not candidates:
        candidates.add(0)

    for shift in sorted(candidates):
        if shift % n != 0:
            continue
        j_shifted = tuple(x + shift for x in j)
        union = sorted(set(i) | set(j_shifted))
        if len(union) != k + 1:
            continue
        if len({residue_repr(z, n) for z in union}) != len(union):
            continue
        X = normalize(tuple(union), n)
        P_X = set(packet(X, n))
        if normalize(i, n) in P_X and normalize(j, n) in P_X:
            return X
    return None


def add_edge(G: Graph, x: Inv, y: Inv) -> None:
    """Add the directed edge x -> y to a graph using ins/outs sets."""
    if x not in G or y not in G:
        raise AssertionError(f"edge endpoint not in graph: {x} -> {y}")
    G[x]["outs"].add(y)
    G[y]["ins"].add(x)


def add_quasi_relation_from_packet(ret: Graph, P_X: list[Inv], I: set[Inv], k: int) -> bool:
    """Add the permanent-poset relation from one quasi packet, if it is quasi.

    Here P_X is a (k+1)-packet in antilex order and I=Inv_k(w).  The
    function returns True exactly when P_X has a two-element adjacent
    intersection with I and hence contributes one quasi-inversion relation.
    """
    members = [p for p in P_X if p in I]
    if len(members) != 2:
        return False
    idx0 = P_X.index(members[0])
    idx1 = P_X.index(members[1])
    if idx1 != idx0 + 1:
        return False
    paper_i = idx0 + 1
    if (k - paper_i) % 2 == 1:
        add_edge(ret, P_X[idx0], P_X[idx1])
    else:
        add_edge(ret, P_X[idx1], P_X[idx0])
    return True


@lru_cache(maxsize=100000)
def permanent_poset(w: Perm, k: int, affine: bool = True) -> Graph:
    """Return the generating relations of P_w(n,k).

    If affine=False, this is the classical finite permanent poset on
    Inv_k(w) with quasi-inversions only from binom([n],k+1).  If
    affine=True, this includes the affine quasi-inversion and congruence
    relations.  The returned graph is not transitively closed.
    """
    n = len(w)
    I = set(inv(w, k))
    full = set(inv(w, k + 1))
    ret: Graph = {x: {"ins": set(), "outs": set()} for x in I}

    if k == 0:
        return ret

    if not affine:
        for X in finite_subsets(n, k + 1):
            if X in full:
                continue
            add_quasi_relation_from_packet(ret, packet(X, n), I, k)
        return ret

    if affine and k == 1:
        raise NotImplementedError(
            "The affine permanent poset P_w(n,1) is infinite.  This finite notebook "
            "does not use affine=True,k=1; use affine=False for the classical finite tests."
        )

    # Congruence relations X < X+v_i if k-i is odd, and X+v_i < X if k-i is even.
    for x in I:
        for idx in range(1, k):  # idx is zero-based first changed coordinate; paper's i is idx.
            y = tuple(list(x[:idx]) + [z + n for z in x[idx:]])
            y = normalize(y, n)
            if y in I and y != x:
                paper_i = idx
                if (k - paper_i) % 2 == 1:
                    add_edge(ret, x, y)
                else:
                    add_edge(ret, y, x)

    # Affine quasi-inversion relations.  Enumerate possible packets by pairs
    # of k-inversions and keep exactly the two-adjacent-intersection packets.
    seen_packets: set[Inv] = set()
    I_list = sorted(I)
    for a_pos, a in enumerate(I_list):
        for b in I_list[a_pos + 1:]:
            if is_congruent(a, b, n):
                continue
            X = common_packet_affine(a, b, n)
            if X is None or X in full or X in seen_packets:
                continue
            if add_quasi_relation_from_packet(ret, packet(X, n), I, k):
                seen_packets.add(X)

    return ret


def G(w: Perm, k: int, R: frozenset[Inv], affine: bool = True) -> Graph:
    """Return G_R for R subset Inv_k(w), with vertex set Inv_{k-1}(w).

    Use affine=False for the finite/classical checks in the reduction lemmas.
    Use affine=True for the affine conjecture checks; this includes congruence
    edges and affine quasi-inversion edges through permanent_poset(w,k-1,True).
    """
    n = len(w)
    vertices = set(inv(w, k - 1))
    graph: Graph = {x: {"ins": set(), "outs": set()} for x in vertices}

    P = permanent_poset(w, k - 1, affine)
    if set(P) != vertices:
        raise AssertionError("permanent_poset vertex set mismatch")
    for x in P:
        for y in P[x]["outs"]:
            add_edge(graph, x, y)

    inv_k = set(inv(w, k))
    if not set(R) <= inv_k:
        raise AssertionError(f"R is not a subset of Inv_{k}(w): w={w}, bad={set(R)-inv_k}")
    for X in inv_k:
        P_X = packet(X, n)
        for idx in range(len(P_X) - 1):
            if X in R:  # full antilex edge
                add_edge(graph, P_X[idx], P_X[idx + 1])
            else:       # full lex edge
                add_edge(graph, P_X[idx + 1], P_X[idx])
    return graph


@lru_cache(maxsize=None)
def permutations(n: int, ell: int) -> frozenset[Perm]:
    """Return all affine permutations in affine S_n of Coxeter length ell."""
    if ell < 0:
        return frozenset()
    if ell == 0:
        return frozenset([tuple(range(1, n + 1))])

    ret: set[Perm] = set()
    for w in permutations(n, ell - 1):
        for i in range(n):
            if i != n - 1 and w[i] < w[i + 1]:
                v = list(w)
                v[i], v[i + 1] = v[i + 1], v[i]
                ret.add(tuple(v))
            elif i == n - 1 and w[-1] < w[0] + n:
                v = list(w)
                v[0], v[-1] = v[-1] - n, v[0] + n
                ret.add(tuple(v))
    return frozenset(ret)


def coxeter_length(w: Perm) -> int:
    """Coxeter length, computed as the number of 2-inversions."""
    return len(inv(w, 2))


def packets_containing(w: Perm, x: Inv) -> list[Inv]:
    """Return all full packets y in Inv_{len(x)+1}(w) with x in P(y)."""
    n = len(w)
    k = len(x)
    return [y for y in inv(w, k + 1) if x in packet(y, n)]


def is_consistent_on(w: Perm, k: int, rev: frozenset[Inv], xs: list[Inv]) -> bool:
    """Check the MSZ prefix/suffix condition for rev on the listed packets xs."""
    n = len(w)
    for x in xs:
        p = packet(x, n)
        flips = []
        for i in range(k):
            if (p[i] in rev) != (p[i + 1] in rev):
                flips.append(i)
        if len(flips) > 1:
            return False
    return True


@lru_cache(maxsize=100000)
def consistent_sets(w: Perm, k: int, affine: bool = True) -> frozenset[frozenset[Inv]]:
    """Return all consistent reversal subsets of Inv_k(w)."""
    ret: set[frozenset[Inv]] = {frozenset()}
    todo: list[frozenset[Inv]] = [frozenset()]
    P = permanent_poset(w, k, affine)
    I = set(inv(w, k))

    while todo:
        curr = todo.pop()
        for iv in I:
            if iv in curr:
                continue
            if not P[iv]["ins"] <= curr:
                continue
            new = frozenset(set(curr) | {iv})
            if new not in ret and is_consistent_on(w, k, new, packets_containing(w, iv)):
                ret.add(new)
                todo.append(new)
    return frozenset(ret)

def poset_to_hasse(P: Poset) -> Poset:
    """Return the transitive reduction of a poset P"""
    ret: Poset = {x: {"ins": set(), "outs": set()} for x in P}
    P = transitive_closure(P)
    for x in P:
        for y in P[x]["outs"]:
            if len(P[x]["outs"] & P[y]["ins"]) == 0:
                ret[x]["outs"].add(y)
                ret[y]["ins"].add(x)
    return ret

def transitive_closure(P: Graph) -> Graph:
    """Return the transitive closure of a directed graph/poset."""
    ret: Graph = {x: {"ins": set(P[x]["ins"]), "outs": set(P[x]["outs"])} for x in P}
    for _ in range(len(P)):
        changed = False
        for X in list(ret):
            outs = list(ret[X]["outs"])
            for Y in outs:
                for Z in ret[Y]["outs"]:
                    if Z not in ret[X]["outs"]:
                        changed = True
                        ret[X]["outs"].add(Z)
                        ret[Z]["ins"].add(X)
        if not changed:
            break
    return ret


def all_linear_extensions(P: Graph) -> set[tuple[Inv, ...]]:
    """Generate all linear extensions of a finite poset P."""
    elements = tuple(P.keys())

    def gen(order: tuple[Inv, ...] = ()):
        if len(order) == len(elements):
            yield order
            return
        used = set(order)
        for x in elements:
            if x not in used and all(p in used for p in P[x]["ins"]):
                yield from gen(order + (x,))

    return set(gen())


@lru_cache(maxsize=10000)
def admissible_orders(w: Perm, k: int, affine: bool = True) -> frozenset[tuple[Inv, ...]]:
    """Generate all k-admissible orders for w."""
    n = len(w)
    P = permanent_poset(w, k, affine)
    J = set(inv(w, k + 1))
    Ank: set[tuple[Inv, ...]] = set()
    for rho in all_linear_extensions(P):
        admissible = True
        for X in J:
            P_X = packet(X, n)
            P_X_set = set(P_X)
            restrict = [y for y in rho if y in P_X_set]
            if restrict != P_X and restrict != P_X[::-1]:
                admissible = False
                break
        if admissible:
            Ank.add(rho)
    return frozenset(Ank)


def lex_unnested(rho: tuple[Inv, ...], n: int, Invs: set[Inv], X: Inv) -> bool:
    """Test if P(X) is lex-unnested in rho."""
    P_X = packet(X, n)
    P_X_set = set(P_X)
    restrictX = [z for z in rho if z in P_X_set]
    if restrictX != P_X[::-1]:
        return False
    for Y in Invs:
        if Y == X:
            continue
        P_Y = packet(Y, n)
        P_Y_set = set(P_Y)
        restrictY = [z for z in rho if z in P_Y_set]
        if restrictY == P_Y[::-1]:
            restrict_XY = [z for z in rho if z in P_X_set or z in P_Y_set]
            if restrict_XY[0] == P_X[-1] and restrict_XY[-1] == P_X[0]:
                return False
    return True


def antilex_unnested(rho: tuple[Inv, ...], n: int, Invs: set[Inv], X: Inv) -> bool:
    """Test if P(X) is antilex-unnested in rho."""
    P_X = packet(X, n)
    P_X_set = set(P_X)
    restrictX = [z for z in rho if z in P_X_set]
    if restrictX != P_X:
        return False
    for Y in Invs:
        if Y == X:
            continue
        P_Y = packet(Y, n)
        P_Y_set = set(P_Y)
        restrictY = [z for z in rho if z in P_Y_set]
        if restrictY == P_Y:
            restrict_XY = [z for z in rho if z in P_X_set or z in P_Y_set]
            if restrict_XY[0] == P_X[0] and restrict_XY[-1] == P_X[-1]:
                return False
    return True


def has_cycle(graph: Graph) -> bool:
    """Depth-first search for directed cycles."""
    visited: set[Inv] = set()
    active: set[Inv] = set()

    def dfs(v: Inv) -> bool:
        visited.add(v)
        active.add(v)
        for u in graph[v]["outs"]:
            if u in active:
                return True
            if u not in visited and dfs(u):
                return True
        active.remove(v)
        return False

    for v in graph:
        if v not in visited and dfs(v):
            return True
    return False


def bfs_distances(graph: Graph, start: Inv) -> dict[Inv, int]:
    """Directed unweighted shortest-path distances from start."""
    dist = {start: 0}
    q = deque([start])
    while q:
        x = q.popleft()
        for y in graph[x]["outs"]:
            if y not in dist:
                dist[y] = dist[x] + 1
                q.append(y)
    return dist


def edge_count(graph: Graph) -> int:
    return sum(len(graph[x]["outs"]) for x in graph)


def clear_structure_caches() -> None:
    """Clear caches for structures depending on w, but keep affine length layers."""
    inv.cache_clear()
    permanent_poset.cache_clear()
    consistent_sets.cache_clear()
    admissible_orders.cache_clear()


def finite_permutation_layers(n: int) -> dict[int, set[Perm]]:
    """Return finite permutations in S_n grouped by Coxeter length."""
    max_len = math.comb(n, 2)
    layers: dict[int, set[Perm]] = {}
    for ell in range(max_len + 1):
        layers[ell] = {w for w in permutations(n, ell) if all(x > 0 for x in w)}
    return layers

def verify_finite_enumeration(max_n: int = 6) -> dict[int, dict[str, int]]:
    """Sanity/completeness check for finite S_n enumeration used below."""
    summary: dict[int, dict[str, int]] = {}
    for n in range(3, max_n + 1):
        layers = finite_permutation_layers(n)
        union = set().union(*layers.values())
        expected = math.factorial(n)
        if len(union) != expected:
            raise AssertionError(f"finite enumeration incomplete for n={n}: got {len(union)}, expected {expected}")
        if sum(len(v) for v in layers.values()) != expected:
            raise AssertionError(f"finite enumeration has duplicate length layers for n={n}")
        for ell, layer in layers.items():
            for w in layer:
                if coxeter_length(w) != ell:
                    raise AssertionError(f"wrong length layer: n={n}, ell={ell}, w={w}, length={coxeter_length(w)}")
        summary[n] = {"finite_permutations": len(union), "length_layers": len(layers)}
        print(f"finite enumeration n={n}: {len(union)}={n}! permutations over {len(layers)} length layers")
    return summary


def verify_partition_edges() -> dict[int, dict[str, int]]:
    """Verification for Lemma partition: classical base cases 3 <= n <= 6."""
    summary: dict[int, dict[str, int]] = {}
    for n in range(3, 7):
        stats = {"permutations": 0, "consistent_sets": 0, "vertices": 0, "edges": 0}
        k = n - 1
        for ell in range(math.comb(n, 2) + 1):
            for w in permutations(n, ell):
                if not all(i > 0 for i in w):
                    continue
                stats["permutations"] += 1
                for R in consistent_sets(w, k, affine=False):
                    stats["consistent_sets"] += 1
                    G_R = G(w, k, R, affine=False)
                    classes: dict[Inv, int] = {}
                    for v in inv(w, k - 1):
                        if v + (n,) in R:
                            classes[v] = 2
                        elif n in v:
                            classes[v] = 1
                        else:
                            classes[v] = 0
                    if set(classes) != set(G_R):
                        raise AssertionError(f"partition classes do not cover vertices: n={n}, w={w}, R={R}")
                    stats["vertices"] += len(G_R)
                    stats["edges"] += edge_count(G_R)
                    for x in G_R:
                        for y in G_R[x]["outs"]:
                            if classes[x] > classes[y]:
                                raise AssertionError(
                                    f"partition edge counterexample: n={n}, w={w}, R={R}, "
                                    f"edge={x}->{y}, classes={classes[x]}>{classes[y]}"
                                )
        if stats["permutations"] != math.factorial(n):
            raise AssertionError(f"partition test was vacuous/incomplete for n={n}: {stats}")
        summary[n] = stats
        print(f"partition edges n={n}: {stats}")
    return summary


def commute_in_Bn2(x: Inv, y: Inv, n: int) -> bool:
    """For k=2 packets, commute iff their 1-packets are disjoint."""
    return len(set(packet(x, n)) & set(packet(y, n))) == 0


def verify_unnested(kind: str) -> dict[str, int]:
    """Verification for the lex/antilex unnested finite S_4 checks."""
    if kind not in {"lex", "antilex"}:
        raise ValueError("kind must be 'lex' or 'antilex'")
    n = 4
    k = 2
    stats = {
        "permutations": 0,
        "admissible_orders": 0,
        "candidate_packets": 0,
        "unnested_packets": 0,
        "between_elements_checked": 0,
    }
    for ell in range(math.comb(n, 2) + 1):
        for w in permutations(n, ell):
            if not all(i > 0 for i in w):
                continue
            stats["permutations"] += 1
            J = set(inv(w, k + 1))
            for rho in admissible_orders(w, k, affine=False):
                stats["admissible_orders"] += 1
                for X in J:
                    stats["candidate_packets"] += 1
                    if kind == "lex":
                        if not lex_unnested(rho, n, J, X):
                            continue
                        P_X = packet(X, n)
                        a = rho.index(P_X[2])
                        b = rho.index(P_X[1])
                        c = rho.index(P_X[0])
                        stats["unnested_packets"] += 1
                        for idx in range(a + 1, b):
                            stats["between_elements_checked"] += 1
                            if not commute_in_Bn2(P_X[2], rho[idx], n):
                                raise AssertionError(f"lex-unnested counterexample between X3 and X2: w={w}, rho={rho}, X={X}, Y={rho[idx]}")
                        for idx in range(b + 1, c):
                            stats["between_elements_checked"] += 1
                            if not commute_in_Bn2(P_X[0], rho[idx], n):
                                raise AssertionError(f"lex-unnested counterexample between X2 and X1: w={w}, rho={rho}, X={X}, Y={rho[idx]}")
                    else:
                        if not antilex_unnested(rho, n, J, X):
                            continue
                        P_X = packet(X, n)
                        a = rho.index(P_X[0])
                        b = rho.index(P_X[1])
                        c = rho.index(P_X[2])
                        stats["unnested_packets"] += 1
                        for idx in range(a + 1, b):
                            stats["between_elements_checked"] += 1
                            if not commute_in_Bn2(P_X[0], rho[idx], n):
                                raise AssertionError(f"antilex-unnested counterexample between X1 and X2: w={w}, rho={rho}, X={X}, Y={rho[idx]}")
                        for idx in range(b + 1, c):
                            stats["between_elements_checked"] += 1
                            if not commute_in_Bn2(P_X[2], rho[idx], n):
                                raise AssertionError(f"antilex-unnested counterexample between X2 and X3: w={w}, rho={rho}, X={X}, Y={rho[idx]}")
    if stats["permutations"] != 24 or stats["admissible_orders"] == 0 or stats["unnested_packets"] == 0:
        raise AssertionError(f"{kind}-unnested test was vacuous/incomplete: {stats}")
    print(f"{kind}-unnested test: {stats}")
    return stats


def verify_diameter() -> dict[int, dict[str, int]]:
    """Verification for Lemma shorten5 using directed BFS distances."""
    summary: dict[int, dict[str, int]] = {}
    k = 3
    for n in range(3, 7):
        stats = {
            "permutations": 0,
            "consistent_sets": 0,
            "graphs": 0,
            "start_vertices": 0,
            "reachable_pairs": 0,
            "max_directed_distance": 0,
        }
        for ell in range(math.comb(n, 2) + 1):
            for w in permutations(n, ell):
                if not all(i > 0 for i in w):
                    continue
                stats["permutations"] += 1
                for R in consistent_sets(w, k, affine=False):
                    stats["consistent_sets"] += 1
                    G_R = G(w, k, R, affine=False)
                    stats["graphs"] += 1
                    for start in inv(w, k - 1):
                        stats["start_vertices"] += 1
                        dist = bfs_distances(G_R, start)
                        stats["reachable_pairs"] += len(dist) - 1
                        if dist:
                            stats["max_directed_distance"] = max(stats["max_directed_distance"], max(dist.values()))
                        for end, d in dist.items():
                            if d > 3:
                                raise AssertionError(f"diameter counterexample: n={n}, w={w}, R={R}, start={start}, end={end}, distance={d}")
        if stats["permutations"] != math.factorial(n) or stats["graphs"] == 0:
            raise AssertionError(f"diameter test was vacuous/incomplete for n={n}: {stats}")
        summary[n] = stats
        print(f"diameter n={n}: {stats}")
    return summary





def verify_affine_table(table: dict[int, int]) -> dict[int, dict[str, int]]:
    """Verify acyclicity of G_R for the Table 1 affine length bounds.

    This tests the remaining computer-verified range: code parameter
    K=4,...,n, i.e. paper graph parameter k=3,...,n-1.  The paper covers
    the other k-values theoretically.
    """
    summary: dict[int, dict[str, int]] = {}
    for n, max_len in table.items():
        stats = {
            "max_length_inclusive": max_len,
            "affine_permutations": 0,
            "consistent_sets": 0,
            "graphs": 0,
            "edges": 0,
            "cycles_found": 0,
        }
        print(f"affine conjecture n={n}: testing all lengths 0..{max_len} and code K=4..{n}", flush=True)
        start_n = time.perf_counter()
        for ell in range(max_len + 1):
            layer = permutations(n, ell)
            stats["affine_permutations"] += len(layer)
            for w in layer:
                actual_len = coxeter_length(w)
                if actual_len != ell:
                    raise AssertionError(f"affine length-layer error: n={n}, ell={ell}, w={w}, actual_length={actual_len}")
                for K in range(4, n + 1):
                    for R in consistent_sets(w, K, affine=True):
                        stats["consistent_sets"] += 1
                        G_R = G(w, K, R, affine=True)
                        stats["graphs"] += 1
                        stats["edges"] += edge_count(G_R)
                        if has_cycle(G_R):
                            stats["cycles_found"] += 1
                            raise AssertionError(f"affine acyclicity counterexample: n={n}, length={ell}, w={w}, K={K}, R={R}")
            print(f"  n={n}, length={ell}: layer={len(layer)}, cumulative_graphs={stats['graphs']}", flush=True)
            clear_structure_caches()
        elapsed = time.perf_counter() - start_n
        if stats["affine_permutations"] == 0 or stats["graphs"] == 0:
            raise AssertionError(f"affine conjecture test was vacuous for n={n}: {stats}")
        summary[n] = stats
        print(f"affine conjecture n={n} PASSED in {elapsed:.2f}s: {stats}", flush=True)
    return summary


AFFINE_TABLE_LENGTHS = {4: 30, 5: 30, 6: 24, 7: 18, 8: 15}

## Enumeration completeness sanity check

This verifies that the finite layers of lengths used in the reduction tests contain exactly all $n!$ finite permutations in $S_n$.


In [2]:
finite_enumeration_summary = verify_finite_enumeration(6)

finite enumeration n=3: 6=3! permutations over 4 length layers
finite enumeration n=4: 24=4! permutations over 7 length layers
finite enumeration n=5: 120=5! permutations over 11 length layers
finite enumeration n=6: 720=6! permutations over 16 length layers


## Examples from paper





In [3]:
w = tuple([-3,-2,8,7])
poset_to_hasse(permanent_poset(w,2))

{(2, 4): {'ins': set(), 'outs': {(1, 4), (2, 7)}},
 (3, 4): {'ins': set(), 'outs': set()},
 (2, 7): {'ins': {(1, 3), (2, 4)}, 'outs': {(1, 7)}},
 (1, 8): {'ins': {(1, 3), (2, 8)}, 'outs': set()},
 (1, 4): {'ins': {(2, 4)}, 'outs': {(1, 7), (2, 8)}},
 (2, 3): {'ins': set(), 'outs': {(1, 3), (2, 8)}},
 (1, 7): {'ins': {(1, 4), (2, 7)}, 'outs': set()},
 (1, 3): {'ins': {(2, 3)}, 'outs': {(1, 8), (2, 7)}},
 (2, 8): {'ins': {(1, 4), (2, 3)}, 'outs': {(1, 8)}}}

In [4]:
R = frozenset([(1,3,4),(2,7,8),(1,7,8)])
Ex1 = G(w,3,R)
Ex1

{(2, 4): {'ins': {(2, 3)}, 'outs': {(1, 4), (1, 8), (2, 7), (2, 8), (3, 4)}},
 (3, 4): {'ins': {(2, 4)}, 'outs': {(1, 4), (1, 8), (2, 8)}},
 (2, 7): {'ins': {(1, 3), (2, 3), (2, 4), (2, 8)}, 'outs': {(1, 7)}},
 (1, 8): {'ins': {(1, 3), (1, 4), (2, 4), (2, 8), (3, 4)}, 'outs': {(1, 7)}},
 (1, 4): {'ins': {(2, 4), (3, 4)}, 'outs': {(1, 3), (1, 7), (1, 8), (2, 8)}},
 (2, 3): {'ins': set(), 'outs': {(1, 3), (1, 7), (2, 4), (2, 7), (2, 8)}},
 (1, 7): {'ins': {(1, 3), (1, 4), (1, 8), (2, 3), (2, 7)}, 'outs': set()},
 (1, 3): {'ins': {(1, 4), (2, 3)}, 'outs': {(1, 7), (1, 8), (2, 7)}},
 (2, 8): {'ins': {(1, 4), (2, 3), (2, 4), (3, 4)}, 'outs': {(1, 8), (2, 7)}}}

In [5]:
poset_to_hasse(Ex1)

{(2, 4): {'ins': {(2, 3)}, 'outs': {(3, 4)}},
 (3, 4): {'ins': {(2, 4)}, 'outs': {(1, 4)}},
 (2, 7): {'ins': {(1, 3), (2, 8)}, 'outs': {(1, 7)}},
 (1, 8): {'ins': {(1, 3), (2, 8)}, 'outs': {(1, 7)}},
 (1, 4): {'ins': {(3, 4)}, 'outs': {(1, 3), (2, 8)}},
 (2, 3): {'ins': set(), 'outs': {(2, 4)}},
 (1, 7): {'ins': {(1, 8), (2, 7)}, 'outs': set()},
 (1, 3): {'ins': {(1, 4)}, 'outs': {(1, 8), (2, 7)}},
 (2, 8): {'ins': {(1, 4)}, 'outs': {(1, 8), (2, 7)}}}

In [6]:
w = tuple([6,4,5,2,3,1])
poset_to_hasse(permanent_poset(w,3))

{(3, 4, 6): {'ins': {(2, 4, 6), (3, 5, 6)}, 'outs': set()},
 (1, 3, 5): {'ins': {(1, 3, 4)}, 'outs': {(1, 2, 5)}},
 (1, 2, 6): {'ins': {(1, 3, 6)}, 'outs': set()},
 (3, 5, 6): {'ins': {(2, 5, 6)}, 'outs': {(3, 4, 6)}},
 (1, 4, 6): {'ins': {(1, 5, 6)}, 'outs': set()},
 (1, 3, 4): {'ins': set(), 'outs': {(1, 2, 4), (1, 3, 5)}},
 (1, 5, 6): {'ins': set(), 'outs': {(1, 4, 6)}},
 (1, 2, 5): {'ins': {(1, 2, 4), (1, 3, 5)}, 'outs': set()},
 (1, 3, 6): {'ins': set(), 'outs': {(1, 2, 6)}},
 (2, 4, 6): {'ins': {(2, 5, 6)}, 'outs': {(3, 4, 6)}},
 (1, 2, 4): {'ins': {(1, 3, 4)}, 'outs': {(1, 2, 5)}},
 (2, 5, 6): {'ins': set(), 'outs': {(2, 4, 6), (3, 5, 6)}}}

In [7]:
R = frozenset([(1,2,5,6),(1,3,5,6)])
Ex2 = G(w,4,R)
Ex2

{(3, 4, 6): {'ins': {(1, 4, 6), (2, 4, 6), (3, 5, 6)}, 'outs': set()},
 (1, 3, 5): {'ins': {(1, 3, 4), (1, 3, 6)}, 'outs': {(1, 2, 5)}},
 (1, 2, 6): {'ins': {(1, 2, 4), (1, 3, 6), (1, 5, 6)},
  'outs': {(1, 2, 5), (1, 4, 6)}},
 (3, 5, 6): {'ins': {(2, 5, 6)}, 'outs': {(1, 5, 6), (3, 4, 6)}},
 (1, 4, 6): {'ins': {(1, 2, 6), (1, 3, 6), (1, 5, 6)},
  'outs': {(2, 4, 6), (3, 4, 6)}},
 (1, 3, 4): {'ins': set(), 'outs': {(1, 2, 4), (1, 3, 5), (1, 3, 6)}},
 (1, 5, 6): {'ins': {(2, 5, 6), (3, 5, 6)},
  'outs': {(1, 2, 6), (1, 3, 6), (1, 4, 6)}},
 (1, 2, 5): {'ins': {(1, 2, 4), (1, 2, 6), (1, 3, 5)}, 'outs': set()},
 (1, 3, 6): {'ins': {(1, 3, 4), (1, 5, 6)},
  'outs': {(1, 2, 6), (1, 3, 5), (1, 4, 6)}},
 (2, 4, 6): {'ins': {(1, 4, 6), (2, 5, 6)}, 'outs': {(3, 4, 6)}},
 (1, 2, 4): {'ins': {(1, 3, 4)}, 'outs': {(1, 2, 5), (1, 2, 6)}},
 (2, 5, 6): {'ins': set(), 'outs': {(1, 5, 6), (2, 4, 6), (3, 5, 6)}}}

## Partition edges test

For every $3 \le n \le 6$, every $w\in S_n$, and every $R\in \mathcal{C}_w(n,n-1)$, each edge $X\to Y$ in $G_R$ has $X\in V_a$, $Y\in V_b$, and $a\le b$.


In [8]:
partition_summary = verify_partition_edges()


partition edges n=3: {'permutations': 6, 'consistent_sets': 17, 'vertices': 51, 'edges': 51}
partition edges n=4: {'permutations': 24, 'consistent_sets': 43, 'vertices': 166, 'edges': 176}
partition edges n=5: {'permutations': 120, 'consistent_sets': 149, 'vertices': 400, 'edges': 405}
partition edges n=6: {'permutations': 720, 'consistent_sets': 761, 'vertices': 813, 'edges': 768}


## Lex-unnested test

For all $w\in S_4$ and $\rho\in \mathcal{A}_w(n,2)$, if $P(X)$ is lex-unnested, then elements between $X_3$ and $X_2$ commute with $X_3$, and elements between $X_2$ and $X_1$ commute with $X_1$. 


In [9]:
lex_unnested_summary = verify_unnested('lex')


lex-unnested test: {'permutations': 24, 'admissible_orders': 66, 'candidate_packets': 112, 'unnested_packets': 35, 'between_elements_checked': 18}


## Antilex-unnested test

For all $w\in S_4$ and $\rho\in \mathcal{A}_w(n,2)$, if $P(X)$ is antilex-unnested, then elements between $X_1$ and $X_2$ commute with $X_1$, and elements between $X_2$ and $X_3$ commute with $X_3$. 


In [10]:
antilex_unnested_summary = verify_unnested('antilex')


antilex-unnested test: {'permutations': 24, 'admissible_orders': 66, 'candidate_packets': 112, 'unnested_packets': 35, 'between_elements_checked': 18}


## Diameter test

For every $n\le 6$, every $w\in S_n$, every $R\in \mathcal{C}_w(n,3)$, and every start vertex, every directed reachable endpoint has shortest directed distance at most `3`.

In [11]:
diameter_summary = verify_diameter()


diameter n=3: {'permutations': 6, 'consistent_sets': 7, 'graphs': 7, 'start_vertices': 12, 'reachable_pairs': 8, 'max_directed_distance': 2}
diameter n=4: {'permutations': 24, 'consistent_sets': 43, 'graphs': 43, 'start_vertices': 166, 'reachable_pairs': 273, 'max_directed_distance': 2}
diameter n=5: {'permutations': 120, 'consistent_sets': 476, 'graphs': 476, 'start_vertices': 3366, 'reachable_pairs': 10088, 'max_directed_distance': 3}
diameter n=6: {'permutations': 720, 'consistent_sets': 9661, 'graphs': 9661, 'start_vertices': 109897, 'reachable_pairs': 523052, 'max_directed_distance': 3}


## Conjecture test

Check acyclicity of $G_R$ for the values in Table 1: $n=4,5,6,7,8$ up to lengths $30,30,24,18,15$ respectively. This covers code parameter `K=4,...,n`, or equivalently paper graph parameter `k=3,...,n-1`.  The other $k$-ranges are covered by theorems in the paper.  

The full affine check can take a long time.  It prints one progress line per length layer and asserts that every generated affine permutation in that layer has Coxeter length equal to the layer number.  Leave `RUN_FULL_AFFINE_TABLE=True` for the paper verification; set it to `False` only for a quick local smoke test while editing the notebook.


In [12]:
RUN_FULL_AFFINE_TABLE = True

if RUN_FULL_AFFINE_TABLE:
    affine_table_summary = verify_affine_table(AFFINE_TABLE_LENGTHS)
else:
    # Smoke test only; this is not the paper verification.
    affine_table_summary = verify_affine_table({4: 5})


affine conjecture n=4: testing all lengths 0..30 and code K=4..4
  n=4, length=0: layer=1, cumulative_graphs=1
  n=4, length=1: layer=4, cumulative_graphs=5
  n=4, length=2: layer=10, cumulative_graphs=15
  n=4, length=3: layer=20, cumulative_graphs=35
  n=4, length=4: layer=34, cumulative_graphs=69
  n=4, length=5: layer=52, cumulative_graphs=121
  n=4, length=6: layer=74, cumulative_graphs=199
  n=4, length=7: layer=100, cumulative_graphs=307
  n=4, length=8: layer=130, cumulative_graphs=457
  n=4, length=9: layer=164, cumulative_graphs=661
  n=4, length=10: layer=202, cumulative_graphs=935
  n=4, length=11: layer=244, cumulative_graphs=1299
  n=4, length=12: layer=290, cumulative_graphs=1789
  n=4, length=13: layer=340, cumulative_graphs=2441
  n=4, length=14: layer=394, cumulative_graphs=3311
  n=4, length=15: layer=452, cumulative_graphs=4483
  n=4, length=16: layer=514, cumulative_graphs=6077
  n=4, length=17: layer=580, cumulative_graphs=8233
  n=4, length=18: layer=650, cumulat

## Final completeness print

The final message is printed only after all enabled verification cells have run without raising an `AssertionError`.


In [13]:
if RUN_FULL_AFFINE_TABLE:
    print('ALL REQUESTED VERIFICATIONS PASSED for the finite checks and the full affine Table 1 range.')
else:
    print('SMOKE TEST PASSED. Set RUN_FULL_AFFINE_TABLE=True for the full paper verification.')


ALL REQUESTED VERIFICATIONS PASSED for the finite checks and the full affine Table 1 range.
